# Project: Did Google's new ranking algorithm help users?

Google ran an experiment. They changed their search ranking algorithm. 50% of users got the old algorithm (Control — Group A), 50% got the new one (Treatment — Group B). We have the click behavior of 10,000 users. Did the new algorithm actually help?

# PHASE 1: Simulating Search Experiment Data

In [11]:
import warnings
warnings.filterwarnings('ignore')

In [12]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

N_USERS        = 10_000
N_DAYS         = 21
QUERIES_RANGE  = (1, 4)

user_ids = np.arange(1, N_USERS + 1)
groups   = np.random.choice(['control', 'treatment'], size=N_USERS)

users_df = pd.DataFrame({
    'user_id' : user_ids,
    'group'   : groups
})

records = []

for _, user in users_df.iterrows():
    n_queries = np.random.randint(*QUERIES_RANGE)

    for q in range(n_queries):
        day = np.random.randint(1, N_DAYS + 1)
        is_treatment = user['group'] == 'treatment'

        ctr = np.random.binomial(1, p=0.55 if is_treatment else 0.48)

        if ctr == 1:
            dwell_time = np.random.normal(
                loc=180 if is_treatment else 150,
                scale=60
            )
            dwell_time = max(5, dwell_time)
        else:
            dwell_time = 0

        bounce = np.random.binomial(1, p=0.25 if is_treatment else 0.35)

        click_position = np.random.choice(
            [1, 2, 3, 4, 5],
            p=[0.55, 0.20, 0.12, 0.08, 0.05] if is_treatment
              else [0.40, 0.25, 0.15, 0.12, 0.08]
        ) if ctr == 1 else None

        records.append({
            'user_id'        : user['user_id'],
            'group'          : user['group'],
            'day'            : day,
            'clicked'        : ctr,
            'dwell_time_sec' : round(dwell_time, 1),
            'bounced'        : bounce,
            'click_position' : click_position
        })

df = pd.DataFrame(records)
df.to_csv('search_experiment.csv', index=False)

print(f"Dataset created: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nGroup split:")
print(df.groupby('group')['user_id'].nunique())
print(f"\nFirst 5 rows:")
print(df.head())

Dataset created: 20013 rows, 7 columns

Group split:
group
control      5013
treatment    4987
Name: user_id, dtype: int64

First 5 rows:
   user_id      group  day  clicked  dwell_time_sec  bounced  click_position
0        1    control    8        0             0.0        1             NaN
1        1    control    6        0             0.0        1             NaN
2        1    control    5        1           130.3        0             1.0
3        2  treatment   21        1           201.6        0             1.0
4        2  treatment   17        1           220.9        0             4.0
